# Velutina v2 - Clasificador con mas especies + calibracion FP=0 (Colab gratis, 0 euros)

Pulsa **Entorno de ejecucion > Ejecutar todo**. Sin cuentas ni claves ni disco local.
Mejora sobre v1 (`train_yolo_colab.ipynb`, intacto): modelo mayor (`yolov8s-cls`), mas fotos,
mas negativos parecidos (Vespula, Bombus, Eristalis), aumento suave y **calibracion FP=0**
que responde: *a que umbral no toca ninguna abeja y cuantas velutinas pilla?*
Cada celda de codigo es idempotente (se puede re-ejecutar) e imprime su progreso.


In [ ]:
!nvidia-smi
!pip install ultralytics -q
import torch
# --- Config v2 (arriba del todo) ---
MODEL = "yolov8s-cls.pt"
IMGSZ = 320
BATCH = 64
HAS_GPU = torch.cuda.is_available()
EPOCHS = 50 if HAS_GPU else 3
DEVICE = 0 if HAS_GPU else "cpu"
print("GPU disponible:", HAS_GPU)
if HAS_GPU:
    print("Nombre GPU:", torch.cuda.get_device_name(0))
else:
    print("Sin GPU: se usara CPU con EPOCHS=3 (modo prueba).")
print(f"Config v2: MODEL={MODEL} EPOCHS={EPOCHS} IMGSZ={IMGSZ} BATCH={BATCH} DEVICE={DEVICE}")


In [ ]:
# b) Descarga GBIF sin cuentas (misma tecnica v1) - objetivos ampliados v2
import csv, os, time, urllib.parse
import requests
SPECIES = {
    "velutina": ("Vespa velutina", 400),
    "abeja": ("Apis mellifera", 400),
    "crabro": ("Vespa crabro", 200),
    "vespula": ("Vespula vulgaris", 200),
    "germanica": ("Vespula germanica", 150),
    "bombus": ("Bombus terrestris", 100),
    "eristalis": ("Eristalis tenax", 100),
}
ATTR = "ATTRIBUTION.csv"
print("GBIF API publica, sin cuentas ni claves. Solo licencias creativecommons.org/licenses/by.")
done = set()
rows = []
if os.path.exists(ATTR):
    with open(ATTR, encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    done = {r.get("file", "") for r in rows}
    print(f"ATTRIBUTION existente: {len(done)} ficheros ya descargados, se reutilizan.")
for clase, (sp, target) in SPECIES.items():
    outdir = os.path.join("data", clase)
    os.makedirs(outdir, exist_ok=True)
    have = len([r for r in rows if r.get("clase") == clase])
    print(f"[{clase}] {sp}: objetivo={target}, ya tenemos={have}")
    offset, got = 0, have
    while got < target:
        q = urllib.parse.quote(sp)
        url = f"https://api.gbif.org/v1/occurrence/search?scientificName={q}&mediaType=StillImage&limit=300&offset={offset}"
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        res = r.json().get("results", [])
        time.sleep(0.3)
        print(f"  offset={offset} resultados={len(res)} got={got}/{target}")
        if not res:
            break
        for occ in res:
            if got >= target:
                break
            for m in occ.get("media", []) or []:
                if got >= target:
                    break
                lic = (m.get("license") or "")
                if "creativecommons.org/licenses/by" not in lic.lower():
                    continue
                img = m.get("identifier")
                if not img or not img.lower().startswith("http"):
                    continue
                fname = f"{clase}_{got:04d}.jpg"
                fpath = os.path.join(outdir, fname)
                if fpath in done or os.path.exists(fpath):
                    continue
                try:
                    im = requests.get(img, timeout=30)
                    im.raise_for_status()
                    open(fpath, "wb").write(im.content)
                except Exception as e:
                    print(f"    fallo imagen: {e}")
                    continue
                rows.append({"file": fpath, "clase": clase, "scientificName": sp, "gbifKey": occ.get("key", ""), "imageUrl": img, "license": lic, "rightsHolder": occ.get("rightsHolder", ""), "creator": m.get("creator", "")})
                done.add(fpath)
                got += 1
                if got % 20 == 0:
                    print(f"    ... {got}/{target}")
        offset += 300
        if offset > 9000:
            break
    if got < 30:
        print(f"AVISO: [{clase}] solo {got}/{target} fotos (<30). Se sigue con las demas especies.")
    else:
        print(f"[{clase}] listo: {got}/{target} en {outdir}")
with open(ATTR, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["file", "clase", "scientificName", "gbifKey", "imageUrl", "license", "rightsHolder", "creator"])
    w.writeheader()
    w.writerows(rows)
print(f"ATTRIBUTION guardado: {ATTR} con {len(rows)} filas")


In [ ]:
# c) Split 80/20 estratificado seed 42 a ds2/train|val/<clase>/ (idempotente)
import os, random, shutil
CLASES = ["velutina", "abeja", "crabro", "vespula", "germanica", "bombus", "eristalis"]
print("Split 80/20 estratificado, seed=42 -> ds2/")
for c in CLASES:
    for split in ["train", "val"]:
        d = os.path.join("ds2", split, c)
        if os.path.isdir(d):
            for f in os.listdir(d):
                src = os.path.join(d, f)
                dst = os.path.join("data", c, f)
                if os.path.isfile(src):
                    os.makedirs(os.path.dirname(dst), exist_ok=True)
                    shutil.move(src, dst)
    print(f"  [{c}] ds2/ vaciado (si existia), todo de vuelta en data/{c}/")
for c in CLASES:
    src = os.path.join("data", c)
    files = sorted([f for f in os.listdir(src) if os.path.isfile(os.path.join(src, f))]) if os.path.isdir(src) else []
    rnd = random.Random(42)
    rnd.shuffle(files)
    n = len(files)
    n_train = int(n * 0.8)
    train_f, val_f = files[:n_train], files[n_train:]
    for split, names in [("train", train_f), ("val", val_f)]:
        d = os.path.join("ds2", split, c)
        os.makedirs(d, exist_ok=True)
        for f in names:
            shutil.move(os.path.join(src, f), os.path.join(d, f))
    print(f"  [{c}] total={n} train={len(train_f)} val={len(val_f)} -> ds2/train/{c}/ ds2/val/{c}/")
print("Split listo. Formato clasificacion por carpetas en ds2/")


## Por que clasificacion y no deteccion (igual que v1)

GBIF da una etiqueta por foto, sin cajas. Sin cajas no hay detector (`yolov8n.pt`).
Entrenamos clasificador (`yolov8s-cls.pt`, mayor que el `yolov8n-cls` de v1): dice
*velutina / abeja / crabro / vespula / germanica / bombus / eristalis* en un recorte.
La deteccion con cajas es la seccion opcional con `kaggle.json` al final.


In [ ]:
# d) Entrenar YOLO clasificador v2 con aumento suave - idempotente
import torch
from ultralytics import YOLO
HAS_GPU = torch.cuda.is_available()
MODEL = "yolov8s-cls.pt"
IMGSZ, BATCH = 320, 64
EPOCHS = 50 if HAS_GPU else 3
DEVICE = 0 if HAS_GPU else "cpu"
print(f"Entrenando {MODEL}: epochs={EPOCHS} imgsz={IMGSZ} batch={BATCH} device={DEVICE}")
print("Aumento suave: degrees=15 translate=0.1 scale=0.2 fliplr=0.5 hsv_h=0.02 hsv_s=0.4")
model = YOLO(MODEL)
model.train(data="ds2", epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=DEVICE, degrees=15, translate=0.1, scale=0.2, fliplr=0.5, hsv_h=0.02, hsv_s=0.4)
print("Entreno listo. Pesos en runs/classify/train/weights/best.pt")


In [ ]:
# e) Evaluar v2: top1 por clase + matriz + resultados_v2.txt (mismo formato v1 + clases nuevas)
import glob, os
from ultralytics import YOLO
model = YOLO("runs/classify/train/weights/best.pt" if os.path.exists("runs/classify/train/weights/best.pt") else "yolov8s-cls.pt")
print("Validando con model.val() ...")
metrics = model.val()
print("top1:", getattr(metrics, "top1", "ver runs/classify/val"))
print("top5:", getattr(metrics, "top5", "ver runs/classify/val"))
cm = glob.glob("runs/classify/val*/confusion_matrix.png") + glob.glob("runs/classify/train/confusion_matrix.png")
print("Matriz confusion:", cm if cm else "revisa carpeta runs/classify/")
clases = ["velutina", "abeja", "crabro", "vespula", "germanica", "bombus", "eristalis"]
stats, ok_tot, n_tot = {}, 0, 0
for c in clases:
    d = os.path.join("ds2", "val", c)
    files = [os.path.join(d, f) for f in sorted(os.listdir(d))] if os.path.isdir(d) else []
    ok = 0
    for fp in files:
        pred = model.predict(fp, verbose=False)[0]
        name = pred.names[pred.probs.top1]
        if name == c:
            ok += 1
        else:
            k = f"FP_predicho_{name}_siendo_{c}"
            stats[k] = stats.get(k, 0) + 1
    acc = ok / max(1, len(files))
    stats[f"top1_{c}"] = round(acc, 4)
    stats[f"n_{c}"] = len(files)
    ok_tot += ok
    n_tot += len(files)
    print(f"  top1 {c}: {acc:.3f} ({ok}/{len(files)})")
top1_global = ok_tot / max(1, n_tot)
print(f"Top1 global: {top1_global:.3f} ({ok_tot}/{n_tot})")
print("Matriz (conteo predicho_X_siendo_Y):")
for k in sorted(stats):
    if k.startswith("FP_predicho"):
        print(f"  {k}: {stats[k]}")
recall_vel = stats.get("top1_velutina", 0)
fp_abeja = stats.get("FP_predicho_velutina_siendo_abeja", 0)
fp_crabro = stats.get("FP_predicho_velutina_siendo_crabro", 0)
print(f"Recall velutina: {recall_vel}")
print(f"FP: abeja->velutina={fp_abeja} crabro->velutina={fp_crabro}")
with open("resultados_v2.txt", "w", encoding="utf-8") as f:
    f.write(f"top1_global: {top1_global:.4f}\n")
    f.write(f"top1: {top1_global:.4f}\n")
    for k in sorted(stats):
        f.write(f"{k}: {stats[k]}\n")
    f.write(f"recall_velutina: {recall_vel}\n")
    f.write(f"FP_abeja_como_velutina: {fp_abeja}\n")
    f.write(f"FP_crabro_como_velutina: {fp_crabro}\n")
    f.write(f"FP_vespula_como_velutina: {stats.get('FP_predicho_velutina_siendo_vespula', 0)}\n")
    f.write(f"FP_germanica_como_velutina: {stats.get('FP_predicho_velutina_siendo_germanica', 0)}\n")
    f.write(f"FP_bombus_como_velutina: {stats.get('FP_predicho_velutina_siendo_bombus', 0)}\n")
    f.write(f"FP_eristalis_como_velutina: {stats.get('FP_predicho_velutina_siendo_eristalis', 0)}\n")
print("Informe guardado en resultados_v2.txt (mismo formato v1 + clases nuevas)")


In [ ]:
# f) Calibracion FP=0 (celda clave): a que umbral no toca ninguna abeja y cuantas pilla?
import json, os
import numpy as np
from ultralytics import YOLO
model = YOLO("runs/classify/train/weights/best.pt" if os.path.exists("runs/classify/train/weights/best.pt") else "yolov8s-cls.pt")
names = model.names
idx_vel = next(i for i, n in names.items() if n == "velutina")
print("names:", dict(names), "idx_velutina:", idx_vel)
clases = ["velutina", "abeja", "crabro", "vespula", "germanica", "bombus", "eristalis"]
pv_vel, pv_neg = [], []
for c in clases:
    d = os.path.join("ds2", "val", c)
    files = [os.path.join(d, f) for f in sorted(os.listdir(d))] if os.path.isdir(d) else []
    for fp in files:
        pred = model.predict(fp, verbose=False)[0]
        pv = float(pred.probs.data.cpu().ravel()[idx_vel])
        (pv_vel if c == "velutina" else pv_neg).append(pv)
    n = len(pv_vel) if c == "velutina" else len(pv_neg)
    print(f"  Pv recogidas [{c}]: acumulado={n}")
print(f"Total: velutina={len(pv_vel)} negativos(abeja+crabro+vespula+germanica+bombus+eristalis)={len(pv_neg)}")
umbrales = [round(float(x), 4) for x in np.linspace(0.5, 0.999, 10)]
print("Tabla umbral/recall_velutina/FP_total (10 filas):")
print(f"{'umbral':>8} {'recall':>8} {'FP':>6}")
best = {"umbral": 0.999, "recall": 0.0, "fp": 0}
for u in umbrales:
    rec = sum(1 for p in pv_vel if p >= u) / max(1, len(pv_vel))
    fp = sum(1 for p in pv_neg if p >= u)
    print(f"{u:>8.4f} {rec:>8.3f} {fp:>6d}")
    if fp == 0:
        best = {"umbral": u, "recall": round(float(rec), 4), "fp": 0}
print(f"Operating point FP=0: MAYOR recall velutina con FP_total=0 -> {best}")
open("operating_point.json", "w", encoding="utf-8").write(json.dumps(best, indent=2))
print("Guardado operating_point.json:", best)


In [ ]:
# g) Exportar a ONNX + tamano best.pt + instrucciones de descarga
import glob, os
from ultralytics import YOLO
pt = "runs/classify/train/weights/best.pt"
model = YOLO(pt if os.path.exists(pt) else "yolov8s-cls.pt")
print("Exportando model.export(format=onnx) ...")
out = model.export(format="onnx")
print("ONNX:", out)
cands = glob.glob("runs/classify/train*/weights/best.pt")
best = sorted(cands)[-1] if cands else None
if best:
    mb = os.path.getsize(best) / 1e6
    print(f"best.pt: {best} ({mb:.1f} MB, ~12MB siendo yolov8s)")
else:
    print("best.pt aun no encontrado, revisa runs/classify/")
print("")
print("DESCARGA (panel Archivos de Colab, click der -> Descargar):")
print("  1. runs/classify/train/weights/best.pt (~12MB, modelo s) -> en tu repo copialo como jetson/ai/yolov8n_velutina_v2.pt")
print("  2. resultados_v2.txt -> guardalo junto al peso para comparar con v1")
print("  3. operating_point.json -> guardalo junto al peso (umbral FP=0)")
print("Comparar con v1: python sim/dataset/eval_yolo.py --weights jetson/ai/yolov8n_velutina_v2.pt --umbral 0.99")


In [ ]:
# h) OPCIONAL solo si subes tu kaggle.json: detector Hornet3000+ con cajas YOLO
import glob, os
cands = ["kaggle.json", os.path.expanduser("~/.config/kaggle/kaggle.json"), "/root/.config/kaggle/kaggle.json"]
kj = next((p for p in cands if os.path.exists(p)), None)
if kj is None and "KAGGLE_USERNAME" not in os.environ:
    print("Sin kaggle.json: salto la seccion opcional del detector (sin error).")
    print("Si quieres: sube tu kaggle.json a Colab (solo si subes tu kaggle.json) y re-ejecuta esta celda.")
else:
    print(f"kaggle.json detectado ({kj or 'env'}): descargando Hornet3000+ ...")
    import shutil
    dst = os.path.expanduser("~/.config/kaggle/kaggle.json")
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    print("kaggle datasets download -d joseortizg/hornet3000 ...")
    os.system("kaggle datasets download -d joseortizg/hornet3000 --unzip -p hornet3000")
    from ultralytics import YOLO
    det = YOLO("yolov8n.pt")
    yamls = glob.glob("hornet3000/**/data.yaml", recursive=True)
    data = yamls[0] if yamls else "hornet3000/data.yaml"
    print(f"Entrenando detector yolov8n.pt 30 epochs con sus cajas YOLO ({data}) ...")
    det.train(data=data, epochs=30, imgsz=640)
    print("Comparativa cls vs det en 5 lineas:")
    print("1. cls dice QUE bicho hay en un recorte; det dice DONDE hay bichos en la escena.")
    print("2. cls se entrena con etiquetas GBIF baratas; det exige cajas YOLO (Hornet3000+).")
    print("3. cls no localiza en vuelo; det si, pero pide mas datos y GPU.")
    print("4. cls v2 calibra FP=0 con operating_point.json; det calibrara confianza por caja.")
    print("5. Para campo: det para localizar + cls v2 para confirmar el recorte.")


## Balance honesto v2

**Que demuestra:** pipeline con mas fotos (~1550), modelo mayor (`yolov8s-cls`), negativos
dificiles (Vespula, Bombus, Eristalis tenax que imita abeja) y punto de operacion FP=0
(`operating_point.json`): a que umbral no toca ninguna abeja y cuantas velutinas pilla.

**Limites:**
- Siguen siendo fotos centradas de GBIF, no vuelo real con tu camara.
- Clasificacion != deteccion: dice *que* hay en el recorte, no *donde* en la escena.
- Crabro/Vespula (y germanica/vulgaris entre si) se parecen mucho: espera confusion entre avispas.
- Eristalis imita abeja a proposito: algun FP abeja/eristalis es normal.
- Sin GPU solo 3 epochs de prueba: los numeros buenos exigen GPU gratis de Colab.
